<a href="https://colab.research.google.com/github/Kanishag/bigdata-sentiment-cloud./blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets pyspark vaderSentiment


In [ ]:
from datasets import load_dataset
ds = load_dataset("bdstar/Tweets-Sentiment-Analysis")
print(ds)
df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
print(df.head())


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim

# find the text column automatically
text_col = next((c for c in df.columns if c.lower() in ["text", "tweet", "sentence", "content"]), None)
print("Text column:", text_col)

# use 300k rows so Colab doesn't crash (still big data)
pdf = df.sample(min(300000, len(df)), random_state=1)[[text_col]].rename(columns={text_col: "text"})

spark = SparkSession.builder.appName("sentiment").getOrCreate()
sdf = spark.createDataFrame(pdf)
print("Raw rows:", sdf.count())

sdf = sdf.dropDuplicates(["text"]).dropna(subset=["text"])

sdf = (sdf
  .withColumn("clean", lower(col("text")))
  .withColumn("clean", regexp_replace("clean", r"http\S+|www\.\S+", ""))
  .withColumn("clean", regexp_replace("clean", r"@\w+", ""))
  .withColumn("clean", regexp_replace("clean", r"[^a-z0-9\s#]", " "))
  .withColumn("clean", regexp_replace("clean", r"\s+", " "))
  .withColumn("clean", trim(col("clean"))))

print("Clean rows:", sdf.count())
sdf.select("text", "clean").show(5, truncate=60)


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
pdf = sdf.select("text", "clean").toPandas()

vader = SentimentIntensityAnalyzer()
def vader_label(t):
    s = vader.polarity_scores(t)["compound"]
    return "positive" if s >= 0.05 else "negative" if s <= -0.05 else "neutral"

pdf["vader"] = pdf["text"].apply(vader_label)
print(pdf["vader"].value_counts())


In [ ]:
!pip install -q transformers torch
from transformers import pipeline
clf = pipeline("sentiment-analysis",
               model="cardiffnlp/twitter-roberta-base-sentiment-latest",
               device=0, truncation=True, max_length=128)

sample = pdf.sample(2000, random_state=1).copy()
out = clf(sample["text"].tolist(), batch_size=32)
sample["roberta"] = [o["label"].lower() for o in out]

print(sample["roberta"].value_counts())
print("Agreement VADER vs RoBERTa:",
      (sample["vader"] == sample["roberta"]).mean())


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import chi2_contingency

# 1. Assign each post a topic by keyword
topics = {
    "work/jobs": ["job", "work", "boss", "unemployed", "career", "hired", "fired"],
    "school": ["school", "exam", "homework", "class", "teacher", "college"],
    "health": ["sick", "doctor", "hospital", "pain", "flu", "health"],
    "money": ["money", "pay", "rent", "broke", "bills", "price"],
}
def get_topic(t):
    for name, words in topics.items():
        if any(re.search(rf"\b{w}\b", t) for w in words):
            return name
    return "other"

pdf["topic"] = pdf["clean"].apply(get_topic)
tp = pdf[pdf["topic"] != "other"]

# 2. Sentiment share per topic
table = pd.crosstab(tp["topic"], tp["vader"], normalize="index") * 100
print(table.round(1))
table.plot(kind="bar", stacked=True, figsize=(7,4))
plt.ylabel("% of posts"); plt.title("Sentiment by topic")
plt.tight_layout(); plt.savefig("sentiment_by_topic.png"); plt.show()

# 3. Statistical test: is sentiment related to topic?
chi2, p, dof, _ = chi2_contingency(pd.crosstab(tp["topic"], tp["vader"]))
print(f"Chi-square = {chi2:.1f}, p-value = {p:.4g}")

# 4. Top words per sentiment
stop = set("the a an and to of i is in it my you for on me so at be this that with was but have are just not im get".split())
for s in ["positive", "negative"]:
    words = " ".join(pdf[pdf["vader"] == s]["clean"]).split()
    top = Counter(w for w in words if w not in stop and len(w) > 2).most_common(10)
    print(s, "->", top)


In [ ]:

!pip install -q pymongo certifi
from pymongo import MongoClient
import certifi
import pandas as pd

MONGO_URI = "hidden"
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
print(client.admin.command("ping"))

client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
print(client.admin.command("ping"))
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db = client["sentiment_project"]

# 1. sentiment share per topic (the table from Step 6)
counts = pd.crosstab(tp["topic"], tp["vader"]).reset_index()
db["topic_sentiment"].delete_many({})
db["topic_sentiment"].insert_many(counts.to_dict("records"))

# 2. overall counts
overall = pdf["vader"].value_counts().reset_index()
overall.columns = ["sentiment", "count"]
db["overall"].delete_many({})
db["overall"].insert_many(overall.to_dict("records"))

# 3. a sample of 5,000 posts for the dashboard
sample_posts = pdf.sample(5000, random_state=1)[["text", "vader", "topic"]]
db["posts"].delete_many({})
db["posts"].insert_many(sample_posts.to_dict("records"))

print(db.list_collection_names())
print(db["topic_sentiment"].count_documents({}), "topic rows saved")

In [ ]:
from pymongo import MongoClient
import certifi

MONGO_URI = "hidden"
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
print(client.admin.command("ping"))